# Tutorial 1 — Mass functions on a frame of discernment

This notebook walks through the core primitives shipped in `carla-evidence` v0.1.0:

1. Building a `Frame` (the frame of discernment $\Theta$).
2. Constructing a `MassFunction` (Basic Belief Assignment / BBA).
3. Querying $m$, $\mathrm{Bel}$, $\mathrm{Pl}$, and $Q$ (commonality).
4. Helper constructors: vacuous, categorical, Bayesian.
5. Theoretical modes: Shafer (1976) vs Smets's TBM (1990).
6. Dense numpy materialisation (`to_dense`, `to_bel_vector`, ...).
7. JSON serialisation round-trip.

Combination rules (Dempster, conjunctive, PCR5/6, cautious, ...) land in v0.2.0. DSmT support arrives at the same time.

In [ ]:
from carla_evidence import Frame, MassFunction

## 1. Frame

A `Frame` is an ordered, hashable tuple of distinct hypotheses. The order fixes the bitmask encoding (element `i` ↔ bit `i`).

In [ ]:
theta = Frame.of("car", "truck", "pedestrian", "cyclist")
print(theta)
print("|Theta| =", len(theta))
print("|2^Theta| =", theta.size)
print("full mask =", theta.full_mask)

In [ ]:
# Subset <-> bitmask translation
print("to_bitmask({car, pedestrian}) =", theta.to_bitmask(("car", "pedestrian")))
print("from_bitmask(5) =", theta.from_bitmask(5))

## 2. Building a mass function

A `MassFunction` distributes evidence over subsets of $\Theta$ such that $\sum_A m(A) = 1$ and $m(A) \geq 0$. Below, the lidar weakly favours "car" but reserves some mass for the pair `{car, truck}` and for total ignorance $\Theta$.

In [ ]:
m_lidar = MassFunction(
    theta,
    {
        ("car",): 0.55,
        ("car", "truck"): 0.30,
        theta.omega: 0.15,
    },
)
m_lidar

In [ ]:
print("focals (canonical sorted by bitmask):", m_lidar.focals)
print("is_normal :", m_lidar.is_normal)
print("is_bayesian :", m_lidar.is_bayesian)
print("is_vacuous :", m_lidar.is_vacuous)

## 3. Belief, plausibility, commonality

$$\mathrm{Bel}(A) = \sum_{\emptyset \neq B \subseteq A} m(B), \quad \mathrm{Pl}(A) = \sum_{B \cap A \neq \emptyset} m(B), \quad Q(A) = \sum_{B \supseteq A} m(B).$$

In [ ]:
for A in [("car",), ("car", "truck"), ("pedestrian",), theta.omega]:
    print(f"A = {A}")
    print(f"  m(A)  = {m_lidar.mass(A):.4f}")
    print(f"  Bel(A)= {m_lidar.bel(A):.4f}")
    print(f"  Pl(A) = {m_lidar.pl(A):.4f}")
    print(f"  Q(A)  = {m_lidar.q(A):.4f}")

## 4. Helper constructors

- `vacuous(frame)`: total ignorance, $m(\Theta) = 1$.
- `categorical(frame, A)`: certainty, $m(A) = 1$.
- `bayesian(frame, probs)`: only singleton focals, masses = probabilities.

In [ ]:
m_vacuous = MassFunction.vacuous(theta)
m_certain_car = MassFunction.categorical(theta, ("car",))
m_bayes = MassFunction.bayesian(theta, [0.7, 0.2, 0.05, 0.05])

print("vacuous       :", m_vacuous)
print("categorical   :", m_certain_car)
print("bayesian      :", m_bayes)

# Bayesian collapses to ordinary probability: Pl({car}) = Bel({car}) = 0.7
print("\nBel({car}) on bayesian =", m_bayes.bel(("car",)))
print("Pl({car}) on bayesian  =", m_bayes.pl(("car",)))

## 5. Modes: Shafer vs TBM

- **Shafer (1976)** enforces $m(\emptyset) = 0$.
- **TBM (Smets 1990)** allows $m(\emptyset) \geq 0$ to encode "open-world" mass.

Trying to build a Shafer BBA with $m(\emptyset) > 0$ raises `ModeError`. DSmT support lands in v0.2.0 alongside PCR5/PCR6.

In [ ]:
theta_small = Frame.of("a", "b")
m_tbm = MassFunction(
    theta_small,
    {(): 0.2, ("a",): 0.5, theta_small.omega: 0.3},
    mode="tbm",
)
print(m_tbm)
print("is_normal       :", m_tbm.is_normal)
print("m(empty-set)    :", m_tbm.mass(()))
print("Bel({a})        :", m_tbm.bel(("a",)))  # = m({a}) = 0.5
print("Pl({a})         :", m_tbm.pl(("a",)))   # = m({a}) + m(Theta) = 0.8
print("Bel(Theta)      :", m_tbm.bel(theta_small.omega))  # = 1 - m(empty-set) = 0.8

## 6. Dense numpy materialisation

Internally, `MassFunction` is sparse (a tuple of `(bitmask, mass)` pairs sorted canonically). Algorithms that need a dense view of $2^{|\Theta|}$ entries can request one via `to_dense`, `to_bel_vector`, `to_pl_vector`, `to_q_vector`.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
print("to_dense       :", m_lidar.to_dense())
print("to_bel_vector  :", m_lidar.to_bel_vector())
print("to_pl_vector   :", m_lidar.to_pl_vector())
print("to_q_vector    :", m_lidar.to_q_vector())

## 7. Serialisation

`to_dict` produces a `{names: mass}` mapping (lossy w.r.t. frame and mode); `to_json` / `from_json` are full round-trips.

In [ ]:
payload = m_lidar.to_json(indent=2)
print(payload)

recovered = MassFunction.from_json(payload)
print("\nround-trip equal:", recovered.is_close_to(m_lidar))

## What's next

- **v0.2.0 (Phase 2)** — combination rules: Dempster, conjunctive, disjunctive, Yager, Dubois-Prade, PCR5, PCR6, cautious / bold (Denoeux 2008), mean. Also brings DSmT mode.
- **v0.3.0 (Phase 3)** — decision transforms (BetP, plausibility transform, max-belief, intervals) and uncertainty metrics (conflict $K$, non-specificity, discord, $AU$, Jousselme distance).
- **v0.4.0 (Phase 4)** — discounting (Shafer classical, Mercier contextual, temporal decay).

See `carla-evidence-architecture.md` §7 for the full roadmap.